# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Complete each part that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [46]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]

device: cuda


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [47]:
def load_mnist(train):
    ds = torchvision.datasets.MNIST("data", train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [48]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [49]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [50]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [51]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [52]:
LAM = {"linf": 0.7, "l2": 0.7}


def propagation_region(model, x, y, eps, lam, norm):
    # Returns (center, tau).
    # TODO: tau = lam * eps; PGD center in the (eps - tau)-ball around x
    model.eval()
    tau = lam * eps
    center = pgd(model, x, y, eps - tau, norm, 5)
    model.train()
    return (center, tau)



def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    # TODO: CE of the negative IBP margin bounds on the small region
    center, tau = propagation_region(model, x, y, eps, lam, norm)
    m = ibp_margin_bounds(model, center, tau, norm, y)
    z_hat = -m
    return F.cross_entropy(z_hat, y, reduction=reduction)
    


def single_norm_loss(model, x, y, t, norm):
    # TODO: standard loss for t < 1, then the certified loss with the scheduled radius
    if t < 1:
        return standard_loss(model, x, y, t)
    else:
        return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)

In [53]:
models, results = {}, []
for name, loss_fn in [("standard", standard_loss),
                      ("linf", lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf")),
                      ("l2", lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))]:
    models[name] = make_model()
    sec = train(models[name], loss_fn)
    results.append(dict(evaluate(models[name], name, attacks=True), time=sec))
print_results(results)

epoch  1  loss 0.5763  time 0 s
epoch  2  loss 0.2039  time 1 s
epoch  3  loss 0.1513  time 1 s
epoch  4  loss 0.1216  time 2 s
epoch  5  loss 0.1044  time 2 s
epoch  6  loss 0.0888  time 2 s
epoch  7  loss 0.0793  time 2 s
epoch  8  loss 0.0691  time 3 s
epoch  9  loss 0.0623  time 3 s
epoch 10  loss 0.0561  time 3 s
epoch 11  loss 0.0507  time 3 s
epoch 12  loss 0.0484  time 4 s
epoch 13  loss 0.0445  time 4 s
epoch 14  loss 0.0387  time 4 s
epoch 15  loss 0.0349  time 4 s
epoch 16  loss 0.0186  time 5 s
epoch 17  loss 0.0147  time 5 s
epoch 18  loss 0.0134  time 5 s
epoch 19  loss 0.0124  time 6 s
epoch 20  loss 0.0119  time 6 s
epoch  1  loss 0.6006  time 0 s
epoch  2  loss 0.5144  time 2 s
epoch  3  loss 0.4979  time 3 s
epoch  4  loss 0.4949  time 6 s
epoch  5  loss 0.5215  time 7 s
epoch  6  loss 0.5536  time 10 s
epoch  7  loss 0.5961  time 11 s
epoch  8  loss 0.6544  time 13 s
epoch  9  loss 0.7225  time 15 s
epoch 10  loss 0.7936  time 16 s
epoch 11  loss 0.8683  time 18 s
ep

| Training Method | Clean (%) | L∞ (%) | L2 (%) | Union (%) | PGD L∞ (%) | PGD L2 (%) | PGD Union (%) | Time/Epoch (s) |
| --------------- | --------- | ------ | ------ | --------- | ---------- | ---------- | ------------- | -------------- |
| Standard        | 97.51     | 0.00   | 0.00   | 0.00      | 1.33       | 10.37      | 1.33          | 0.3            |
| L∞              | 93.06     | 79.53  | 4.92   | 4.92      | 83.78      | 70.79      | 70.42         | 1.5            |
| L2              | 92.35     | 54.31  | 58.59  | 51.25     | 69.77      | 78.51      | 69.27         | 1.8            |

# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [54]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    # TODO: per-image max of the two certified losses + eta * KL on the l2-certified subset
    # 1. find all z_hat
    z_hat = {}
    for p in NORMS:
        eps = eps_scale(t) * EPS[p]
        center, tau = propagation_region(model, x, y, eps, LAM[p], p)
        z_hat[p] = -ibp_margin_bounds(model, center, tau, p, y)
    
    # 2. max of the 2 loss
    ce_inf = F.cross_entropy(z_hat["linf"], y, reduction="none")
    ce_2 = F.cross_entropy(z_hat["l2"], y, reduction="none")
    loss = torch.maximum(ce_inf, ce_2).mean()

    if eta <= 0:
        return loss
    
    # 3. the set of images in the batch that are certified for ℓ2
    z2 = z_hat["l2"].detach().clone()
    z2[torch.arange(len(y)), y] = -float("inf")
    l2_certified_mask = z2.max(dim=1).values < 0

    if not l2_certified_mask.any():
        return loss
    
    # 4. KL
    log_d2 = F.log_softmax(z_hat["l2"][l2_certified_mask], dim=1)
    log_dinf = F.log_softmax(z_hat["linf"][l2_certified_mask], dim=1)
    kl = (log_d2.exp() * (log_d2 - log_dinf)).sum(dim=1).mean()

    loss = loss + eta * kl

    return loss

# From hw3
def gp(g_n, g_a):
    # TODO: GP(g_n, g_a) for one parameter tensor
    cosine = F.cosine_similarity(g_n.flatten(), g_a.flatten(), dim=0)
    if cosine > 0:
        return cosine * g_n
    else:
        return torch.zeros_like(g_n)

def gradient_projection(g_n, g_c, beta):
    # TODO: beta * GP(g_n, g_c) + (1 - beta) * g_c for one parameter tensor
    return beta * gp(g_n, g_c) + (1 - beta) * g_c

In [55]:
results = results[:3]
runs = {}
configs = [("cure_max",       lambda m, x, y, t: cure_loss(m, x, y, t, eta=0.0), None),
           ("cure_max_ba",    cure_loss, None),
           ("cure_max_ba_gp", cure_loss, 0.5)]

for seed in (0, 1):
    for name, loss_fn, beta in configs:
        torch.manual_seed(seed)
        model = make_model()
        sec = train(model, loss_fn, gp_beta=beta, seed=seed, verbose=False)
        runs[(name, seed)] = dict(evaluate(model, f"{name} (seed {seed})", attacks=True), time=sec)
        models[(name, seed)] = model
        print_results([runs[(name, seed)]])

for name, _, _ in configs:
    r0, r1 = runs[(name, 0)], runs[(name, 1)]
    mean = {"name": f"{name} (mean)"}
    mean.update({k: (r0[k] + r1[k]) / 2 for k in r0 if k != "name"})
    results.append(mean)

print("\n All results:")
print_results(results)

cure_max (seed 0)         clean  91.90  linf  66.76  l2  56.15  union  56.09  pgd_linf  76.93  pgd_l2  78.45  pgd_union  76.30  time/epoch   3.3 s
cure_max_ba (seed 0)      clean  91.42  linf  65.68  l2  56.06  union  56.00  pgd_linf  75.65  pgd_l2  77.51  pgd_union  75.20  time/epoch   3.4 s
cure_max_ba_gp (seed 0)   clean  91.82  linf  65.58  l2  55.99  union  55.87  pgd_linf  76.21  pgd_l2  77.62  pgd_union  75.66  time/epoch   3.7 s
cure_max (seed 1)         clean  91.44  linf  65.05  l2  56.01  union  55.95  pgd_linf  76.73  pgd_l2  78.48  pgd_union  76.29  time/epoch   4.2 s
cure_max_ba (seed 1)      clean  91.54  linf  65.74  l2  55.87  union  55.80  pgd_linf  76.77  pgd_l2  78.52  pgd_union  76.26  time/epoch   3.5 s
cure_max_ba_gp (seed 1)   clean  92.21  linf  65.65  l2  55.66  union  55.61  pgd_linf  76.59  pgd_l2  78.37  pgd_union  76.01  time/epoch   3.7 s

 All results:
standard                  clean  97.61  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.07  pgd_l2  

# (c) Your method

Write the loss of your method in `my_loss`. Train it with `train` and the default settings, as for the other methods.
If your method needs a different update, write your own copy of `train`.
Do not change the verifier or `evaluate`.

In [56]:
def my_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)

    # 1. find all z_hat
    z_hat = {}
    for p in NORMS:
        eps = eps_scale(t) * EPS[p]
        center, tau = propagation_region(model, x, y, eps, LAM[p], p)
        z_hat[p] = -ibp_margin_bounds(model, center, tau, p, y)
    
    # 2. max of the 2 loss
    ce_inf = F.cross_entropy(z_hat["linf"], y, reduction="none")
    ce_2 = F.cross_entropy(z_hat["l2"], y, reduction="none")
    loss = torch.maximum(ce_inf, ce_2).mean()

    if eta <= 0:
        return loss
    
    # 3. the set of images in the batch that are certified for ℓinf
    zinf = z_hat["linf"].detach().clone()
    zinf[torch.arange(len(y)), y] = -float("inf")
    linf_certified_mask = zinf.max(dim=1).values < 0

    if not linf_certified_mask.any():
        return loss
    
    # 4. KL
    log_d2 = F.log_softmax(z_hat["l2"][linf_certified_mask], dim=1)
    log_dinf = F.log_softmax(z_hat["linf"][linf_certified_mask], dim=1)
    kl = (log_dinf.exp() * (log_dinf - log_d2)).sum(dim=1).mean()

    loss = loss + eta * kl

    return loss

In [57]:
results = results[:3]
runs = {}
configs = [("cure_max_v2",       lambda m, x, y, t: my_loss(m, x, y, t, eta=0.0), None),
           ("cure_max_ba_v2",    my_loss, None),
           ("cure_max_ba_gp_v2", my_loss, 0.5)]

for seed in (0, 1):
    for name, loss_fn, beta in configs:
        torch.manual_seed(seed)
        model = make_model()
        sec = train(model, loss_fn, gp_beta=beta, seed=seed, verbose=False)
        runs[(name, seed)] = dict(evaluate(model, f"{name} (seed {seed})", attacks=True), time=sec)
        models[(name, seed)] = model
        print_results([runs[(name, seed)]])

for name, _, _ in configs:
    r0, r1 = runs[(name, 0)], runs[(name, 1)]
    mean = {"name": f"{name} (mean)"}
    mean.update({k: (r0[k] + r1[k]) / 2 for k in r0 if k != "name"})
    results.append(mean)

print("\n All results:")
print_results(results)

cure_max_v2 (seed 0)      clean  91.90  linf  66.76  l2  56.15  union  56.09  pgd_linf  76.93  pgd_l2  78.45  pgd_union  76.30  time/epoch   3.3 s
cure_max_ba_v2 (seed 0)   clean  91.50  linf  66.12  l2  56.52  union  56.43  pgd_linf  75.96  pgd_l2  77.90  pgd_union  75.55  time/epoch   3.5 s
cure_max_ba_gp_v2 (seed 0)clean  91.89  linf  65.66  l2  56.42  union  56.35  pgd_linf  75.99  pgd_l2  78.10  pgd_union  75.60  time/epoch   3.8 s
cure_max_v2 (seed 1)      clean  91.44  linf  65.05  l2  56.01  union  55.95  pgd_linf  76.73  pgd_l2  78.48  pgd_union  76.29  time/epoch   3.3 s
cure_max_ba_v2 (seed 1)   clean  91.80  linf  65.70  l2  56.93  union  56.83  pgd_linf  76.20  pgd_l2  78.00  pgd_union  75.82  time/epoch   3.5 s
cure_max_ba_gp_v2 (seed 1)clean  92.06  linf  65.32  l2  56.47  union  56.36  pgd_linf  75.34  pgd_l2  77.27  pgd_union  74.89  time/epoch   3.6 s

 All results:
standard                  clean  97.61  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.07  pgd_l2  

| Training Method                  | Clean (%) | L∞ (%) | L2 (%) | Union (%) | PGD L∞ (%) | PGD L2 (%) | PGD Union (%) | Time/Epoch (s) |
| -------------------------------- | --------- | ------ | ------ | --------- | ---------- | ---------- | ------------- | -------------- |
| CURE Max (mean) — L2 based       | 91.67     | 65.91  | 56.08  | 56.02     | 76.83      | 78.47      | 76.30         | 2.9            |
| CURE Max (mean) — L∞ based       | 91.67     | 65.91  | 56.08  | 56.02     | 76.83      | 78.47      | 76.30         | 4.0            |
| CURE Max BA (mean) — L2 based    | 91.48     | 65.71  | 55.97  | 55.90     | 76.21      | 78.02      | 75.73         | 3.2            |
| CURE Max BA (mean) — L∞ based    | 91.65     | 65.91  | 56.73  | 56.63     | 76.08      | 77.95      | 75.69         | 3.5            |
| CURE Max BA GP (mean) — L2 based | 92.01     | 65.62  | 55.83  | 55.74     | 76.40      | 78.00      | 75.84         | 3.4            |
| CURE Max BA GP (mean) — L∞ based | 91.97     | 65.49  | 56.45  | 56.36     | 75.66      | 77.69      | 75.25         | 4.6            |

In [58]:
teacher = make_model()
teacher_time = train(teacher, lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf"))
print(f"Trained linf teacher: {teacher_time:.1f} s/epoch")

teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

print_results([evaluate(teacher, "teacher (linf)")])

epoch  1  loss 0.5763  time 0 s
epoch  2  loss 0.4809  time 2 s
epoch  3  loss 0.4864  time 4 s
epoch  4  loss 0.5056  time 6 s
epoch  5  loss 0.5340  time 8 s
epoch  6  loss 0.5772  time 10 s
epoch  7  loss 0.6283  time 12 s
epoch  8  loss 0.6877  time 14 s
epoch  9  loss 0.7489  time 15 s
epoch 10  loss 0.8191  time 17 s
epoch 11  loss 0.8955  time 19 s
epoch 12  loss 0.9053  time 21 s
epoch 13  loss 0.8634  time 22 s
epoch 14  loss 0.8344  time 24 s
epoch 15  loss 0.8102  time 26 s
epoch 16  loss 0.7437  time 27 s
epoch 17  loss 0.7288  time 29 s
epoch 18  loss 0.7243  time 31 s
epoch 19  loss 0.7217  time 33 s
epoch 20  loss 0.7173  time 34 s
Trained linf teacher: 1.7 s/epoch
teacher (linf)            clean  93.69  linf  79.05  l2   3.09  union   3.09


In [70]:
def my_loss_v3(model, x, y, t, eta=0.5, mu=1.0):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    # TODO: per-image max of the two certified losses + eta * KL on the l2-certified subset
    # 1. find all z_hat
    z_hat, region = {}, {}
    for p in NORMS:
        eps = eps_scale(t) * EPS[p]
        center, tau = propagation_region(model, x, y, eps, LAM[p], p)
        region[p] = (center, tau)
        z_hat[p] = -ibp_margin_bounds(model, center, tau, p, y)
    
    # 2. max of the 2 loss
    ce_inf = F.cross_entropy(z_hat["linf"], y, reduction="none")
    ce_2 = F.cross_entropy(z_hat["l2"], y, reduction="none")
    loss = torch.maximum(ce_inf, ce_2).mean()
    
    # 3. Certified masks
    idx = torch.arange(len(y))
    def cert_mask(z):
        z = z.detach().clone()
        z[idx, y] = -float("inf")
        return z.max(dim=1).values < 0
    l2_cert = cert_mask(z_hat["l2"])
    linf_cert = cert_mask(z_hat["linf"])

    # 4. KL
    if eta > 0 and l2_cert.any():
        log_d2 = F.log_softmax(z_hat["l2"][l2_cert], dim=1)
        log_dinf = F.log_softmax(z_hat["linf"][l2_cert], dim=1)
        loss = loss + eta * (log_d2.exp() * (log_d2 - log_dinf)).sum(dim=1).mean()

    # 4. Teacher distillation: student l2-certified but NOT linf-certified. Teacher is linf-certified on the student's linf small ball
    center, tau = region["linf"]
    with torch.no_grad():
        z_t = -ibp_margin_bounds(teacher, center, tau, "linf", y)
    teacher_cert = cert_mask(z_t)
    mask = l2_cert & ~linf_cert & teacher_cert
    if mask.any():
        log_dt = F.log_softmax(z_t[mask], dim=1)
        log_ds = F.log_softmax(z_hat["linf"][mask], dim=1)
        loss = loss + mu * (log_dt.exp() * (log_dt - log_ds)).sum(dim=1).mean()

    return loss

In [71]:
results = results[:3]
runs = {}
configs = [("cure_max_v3",       lambda m, x, y, t: my_loss_v3(m, x, y, t, eta=0.0), None),
           ("cure_max_ba_v3",    my_loss_v3, None),
           ("cure_max_ba_gp_v3", my_loss_v3, 0.5)]

for seed in (0, 1):
    for name, loss_fn, beta in configs:
        torch.manual_seed(seed)
        model = make_model()
        sec = train(model, loss_fn, gp_beta=beta, seed=seed, verbose=False)
        runs[(name, seed)] = dict(evaluate(model, f"{name} (seed {seed})", attacks=True), time=sec)
        models[(name, seed)] = model
        print_results([runs[(name, seed)]])

for name, _, _ in configs:
    r0, r1 = runs[(name, 0)], runs[(name, 1)]
    mean = {"name": f"{name} (mean)"}
    mean.update({k: (r0[k] + r1[k]) / 2 for k in r0 if k != "name"})
    results.append(mean)

print("\n All results:")
print_results(results)

cure_max_v3 (seed 0)      clean  85.59  linf  57.17  l2  48.28  union  48.27  pgd_linf  67.06  pgd_l2  68.43  pgd_union  66.42  time/epoch   3.9 s
cure_max_ba_v3 (seed 0)   clean  87.42  linf  58.86  l2  49.92  union  49.88  pgd_linf  69.40  pgd_l2  70.84  pgd_union  68.72  time/epoch   4.1 s
cure_max_ba_gp_v3 (seed 0)clean  87.88  linf  58.20  l2  49.05  union  48.96  pgd_linf  68.85  pgd_l2  70.36  pgd_union  68.14  time/epoch   4.5 s
cure_max_v3 (seed 1)      clean  84.25  linf  53.79  l2  46.31  union  46.26  pgd_linf  65.77  pgd_l2  67.01  pgd_union  64.90  time/epoch   4.1 s
cure_max_ba_v3 (seed 1)   clean  79.23  linf  53.60  l2  45.94  union  45.89  pgd_linf  64.14  pgd_l2  65.73  pgd_union  63.56  time/epoch   4.0 s
cure_max_ba_gp_v3 (seed 1)clean  81.63  linf  53.73  l2  45.74  union  45.68  pgd_linf  64.19  pgd_l2  66.37  pgd_union  63.69  time/epoch   4.2 s

 All results:
standard                  clean  97.61  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.07  pgd_l2  

In [72]:
l2_teacher = make_model()
l2_teacher_time = train(l2_teacher, lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"))
print(f"Trained linf teacher: {l2_teacher_time:.1f} s/epoch")

l2_teacher.eval()
for p in l2_teacher.parameters():
    p.requires_grad_(False)

print_results([evaluate(l2_teacher, "l2_teacher (l2)")])

epoch  1  loss 0.5763  time 0 s
epoch  2  loss 0.4562  time 3 s
epoch  3  loss 0.4992  time 5 s
epoch  4  loss 0.5374  time 7 s
epoch  5  loss 0.5898  time 9 s
epoch  6  loss 0.6591  time 10 s
epoch  7  loss 0.7339  time 12 s
epoch  8  loss 0.8261  time 14 s
epoch  9  loss 0.9316  time 15 s
epoch 10  loss 1.0519  time 17 s
epoch 11  loss 1.1714  time 19 s
epoch 12  loss 1.2193  time 20 s
epoch 13  loss 1.1929  time 22 s
epoch 14  loss 1.1712  time 24 s
epoch 15  loss 1.1588  time 25 s
epoch 16  loss 1.1155  time 27 s
epoch 17  loss 1.1049  time 29 s
epoch 18  loss 1.0988  time 30 s
epoch 19  loss 1.0957  time 32 s
epoch 20  loss 1.0945  time 34 s
Trained linf teacher: 1.7 s/epoch
l2_teacher (l2)           clean  92.85  linf  55.17  l2  58.80  union  52.01


In [73]:
def my_loss_v4(model, x, y, t, eta=0.5, mu=1.0):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    # 1. find all z_hat
    z_hat, region = {}, {}
    for p in NORMS:
        eps = eps_scale(t) * EPS[p]
        center, tau = propagation_region(model, x, y, eps, LAM[p], p)
        region[p] = (center, tau)
        z_hat[p] = -ibp_margin_bounds(model, center, tau, p, y)
    
    # 2. max of the 2 loss
    ce_inf = F.cross_entropy(z_hat["linf"], y, reduction="none")
    ce_2 = F.cross_entropy(z_hat["l2"], y, reduction="none")
    loss = torch.maximum(ce_inf, ce_2).mean()
    
    # 3. Certified masks
    idx = torch.arange(len(y))
    def cert_mask(z):
        z = z.detach().clone()
        z[idx, y] = -float("inf")
        return z.max(dim=1).values < 0
    l2_cert = cert_mask(z_hat["l2"])
    linf_cert = cert_mask(z_hat["linf"])

    # 4. KL
    if eta > 0 and linf_cert.any():
        log_d2 = F.log_softmax(z_hat["l2"][linf_cert], dim=1)
        log_dinf = F.log_softmax(z_hat["linf"][linf_cert], dim=1)
        loss = loss + eta * (log_dinf.exp() * (log_dinf - log_d2)).sum(dim=1).mean()

    # 4. Teacher distillation: student linf-certified but NOT l2-certified. Teacher is l2-certified on the student's linf small ball
    center, tau = region["l2"]
    with torch.no_grad():
        z_t = -ibp_margin_bounds(l2_teacher, center, tau, "l2", y)
    teacher_cert = cert_mask(z_t)
    mask = linf_cert & ~l2_cert & teacher_cert
    if mask.any():
        log_dt = F.log_softmax(z_t[mask], dim=1)
        log_ds = F.log_softmax(z_hat["l2"][mask], dim=1)
        loss = loss + mu * (log_dt.exp() * (log_dt - log_ds)).sum(dim=1).mean()

    return loss

In [74]:
results = results[:3]
runs = {}
configs = [("cure_max_v4",       lambda m, x, y, t: my_loss_v4(m, x, y, t, eta=0.0), None),
           ("cure_max_ba_v4",    my_loss_v4, None),
           ("cure_max_ba_gp_v4", my_loss_v4, 0.5)]

for seed in (0, 1):
    for name, loss_fn, beta in configs:
        torch.manual_seed(seed)
        model = make_model()
        sec = train(model, loss_fn, gp_beta=beta, seed=seed, verbose=False)
        runs[(name, seed)] = dict(evaluate(model, f"{name} (seed {seed})", attacks=True), time=sec)
        models[(name, seed)] = model
        print_results([runs[(name, seed)]])

for name, _, _ in configs:
    r0, r1 = runs[(name, 0)], runs[(name, 1)]
    mean = {"name": f"{name} (mean)"}
    mean.update({k: (r0[k] + r1[k]) / 2 for k in r0 if k != "name"})
    results.append(mean)

print("\n All results:")
print_results(results)

cure_max_v4 (seed 0)      clean  87.59  linf  59.51  l2  52.66  union  52.29  pgd_linf  66.84  pgd_l2  69.84  pgd_union  66.26  time/epoch   3.6 s
cure_max_ba_v4 (seed 0)   clean  87.62  linf  60.21  l2  52.38  union  52.08  pgd_linf  67.64  pgd_l2  70.04  pgd_union  67.11  time/epoch   4.3 s
cure_max_ba_gp_v4 (seed 0)clean  88.26  linf  59.95  l2  52.50  union  52.05  pgd_linf  67.60  pgd_l2  70.27  pgd_union  67.11  time/epoch   4.2 s
cure_max_v4 (seed 1)      clean  84.77  linf  56.10  l2  51.67  union  50.88  pgd_linf  62.19  pgd_l2  65.79  pgd_union  61.89  time/epoch   3.9 s
cure_max_ba_v4 (seed 1)   clean  88.44  linf  59.46  l2  52.70  union  52.23  pgd_linf  68.50  pgd_l2  71.00  pgd_union  68.00  time/epoch   3.9 s
cure_max_ba_gp_v4 (seed 1)clean  88.70  linf  59.13  l2  51.92  union  51.53  pgd_linf  68.22  pgd_l2  70.68  pgd_union  67.59  time/epoch   3.8 s

 All results:
standard                  clean  97.61  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.07  pgd_l2  